In [1]:
# Setup paths

import os
import sys

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.append(project_root)

In [2]:
# Imports

from sentence_transformers import SentenceTransformer

from src.config import PDF_PATH, CHUNK_SIZE, CHUNK_OVERLAP, EMBEDDING_MODEL
from src.pdf_utils import extract_text
from src.text_utils import clean_text, chunk_text
from src.vector_store import build_index, search_index
from src.rag import retrieve
from src.embeddings import create_embeddings

In [3]:
# Load pdf --> text --> chunks

text = extract_text(PDF_PATH)
cleaned_text = clean_text(text)

chunks = chunk_text(
    cleaned_text,
    chunk_size=CHUNK_SIZE,
    overlap=CHUNK_OVERLAP
)

print("Total chunks:", len(chunks))

Total chunks: 122


In [4]:
# Load embedding model

model = SentenceTransformer(EMBEDDING_MODEL)

In [5]:
# Create embeddings

import numpy as np

embeddings = create_embeddings(chunks, model)
embeddings = np.ascontiguousarray(embeddings)

print("Embeddings shape:", embeddings.shape)
# (number_of_chunks, 384)

Embeddings shape: (122, 384)


In [6]:
# Building FAISS index

index = build_index(embeddings)
print("FAISS index built")

FAISS index built


In [7]:
query = "What is an entity set?"

In [8]:
# Query embedding

query_embedding = model.encode([query])

query_embedding = np.array(query_embedding).astype("float32")
query_embedding = np.ascontiguousarray(query_embedding)

In [9]:
# Search FAISS

distances, indices = search_index(
    index,
    query_embedding,
    top_k=3
)

In [10]:
# Chunks

for rank, idx in enumerate(indices[0]):
    print("=" * 80)
    print(f"Rank: {rank + 1}")
    print(f"Chunk Index: {idx}")
    print(f"Distance: {distances[0][rank]:.4f}")
    print()
    print(chunks[idx])
    print()

Rank: 1
Chunk Index: 13
Distance: 0.8220

 3. Entity: An Entity is a “thing” or “object” in the real world that is distinguishable from all other objects. 1. It has physical existence. 2. Each student in a college is an entity. 3. Entity can be uniquely identified. (By a primary attribute, aka Primary Key) 4. Strong Entity: Can be uniquely identified. 5. Weak Entity: Can’t be uniquely identified., depends on some other strong entity. 1. It doesn’t have suﬃcient attributes, to select a uniquely identi fiable attribute. 2. Loan -> Strong Entity, Payment -> Weak, as instalments are sequential number counter can be generated separate for each loan. 3. Weak entity depends on strong entity for existence. 4. Entity set 1. It is a set of entities of the same type that share the same properties, or a ttributes. 2. E.g., Student is an entit

Rank: 2
Chunk Index: 20
Distance: 0.9907

some way with other entity sets. 2. Specialisation is splitting up the entity set into further sub entity sets on 

In [11]:
context, results = retrieve(
    query=query,
    index=index,
    chunks=chunks,
    model=model
)

In [12]:
# The best chunk

best_chunk = results[0]

print(best_chunk)

 3. Entity: An Entity is a “thing” or “object” in the real world that is distinguishable from all other objects. 1. It has physical existence. 2. Each student in a college is an entity. 3. Entity can be uniquely identified. (By a primary attribute, aka Primary Key) 4. Strong Entity: Can be uniquely identified. 5. Weak Entity: Can’t be uniquely identified., depends on some other strong entity. 1. It doesn’t have suﬃcient attributes, to select a uniquely identi fiable attribute. 2. Loan -> Strong Entity, Payment -> Weak, as instalments are sequential number counter can be generated separate for each loan. 3. Weak entity depends on strong entity for existence. 4. Entity set 1. It is a set of entities of the same type that share the same properties, or a ttributes. 2. E.g., Student is an entit
